[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day3_Remote_Sensing_Fundamentals_and_Indices/04_digital_image_processing_in_gee.ipynb)

# Day 3 - Fundamentals of Remote Sensing & Spectral Indices
## Session 4: Digital Image Processing in GEE

| | |
|---|---|
| **Fecha / Date** | 2026-10-14 (Wednesday / Miercoles) |
| **Horario / Time** | 15:40 - 17:10 |
| **Entidad / Entity** | UoM |
| **Instructores / Instructors** | Christhian, Sebastian |

### Contenido / Content

> Preparing an image **data cube** in Google Earth Engine: study area, Landsat 8/9 collection,
> cloud filtering, scale factor, cloud masking, spectral indices as new bands, zonal
> statistics over a region of interest, NDVI variability in time, and threshold masks.

### Objetivos de aprendizaje / Learning objectives

- [ ] Build a clean Landsat 8 + 9 collection: filter, scale and remove cloudy pixels
- [ ] Write the indices **NDVI, MSAVI, EVI, NDWI and NDMI** as functions and add them as bands
- [ ] Extract zonal statistics (min, mean, max, standard deviation) for a region drawn on the
      map and read them as a `DataFrame` and as a chart
- [ ] Understand that a pixel value **changes in time** (NDVI min, max and amplitude)
- [ ] Create simple threshold masks for vegetation and water

### Datos necesarios / Required data

No local files. `LANDSAT/LC08/C02/T1_L2` and `LANDSAT/LC09/C02/T1_L2` (Collection 2, Level-2
surface reflectance) from the Earth Engine catalogue.

### Por que esto importa / Why this matters

Everything done here - clean pixels, physical units, extra bands with indices - is the
preparation of the **data cube** (x, y, bands, time) that we will use later for **image
classification**. A classifier is only as good as the bands we give it.

---

## 4.0 - Setup

In [ ]:
# Libraries (uncomment the pip line on a fresh runtime, e.g. Google Colab)
#%pip install -q earthengine-api geemap

import ee                         # Google Earth Engine API
import geemap                     # interactive maps for Earth Engine
import pandas as pd               # tables (DataFrames)
import matplotlib.pyplot as plt   # charts

In [ ]:
# Authenticate and initialise Earth Engine
# >>> REPLACE with your own Cloud project id <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

---

## 4.1 - Area de estudio y coleccion / Study area and image collection

The study area is a strip of the Bolivian altiplano from the shore of **Lago Menor**, across
the **Katari** plain, to the edge of **El Alto**. It contains water, cropland, bare soil and
urban areas - very different surfaces, which makes the indices easy to compare.

### Landsat 8 and 9, Collection 2, Level-2

Both satellites carry almost identical sensors (OLI and OLI-2), so we can **merge** them into
a single collection and get an image every ~8 days instead of every 16.

| Band | Name we will use | Wavelength |
|---|---|---|
| SR_B2 | Blue | 0.45 - 0.51 um |
| SR_B3 | Green | 0.53 - 0.59 um |
| SR_B4 | Red | 0.64 - 0.67 um |
| SR_B5 | NIR | 0.85 - 0.88 um |
| SR_B6 | SWIR1 | 1.57 - 1.65 um |
| SR_B7 | SWIR2 | 2.11 - 2.29 um |

In [ ]:
# Study area (AOI): a rectangle [west, south, east, north]
AOI = ee.Geometry.Rectangle([-68.90, -16.62, -68.10, -16.30])

# Show the AOI on a map
Map = geemap.Map()
Map.centerObject(AOI, 10)
Map.addLayer(AOI, {"color": "red"}, "AOI")
Map

In [ ]:
# Merge Landsat 8 and Landsat 9 and filter by area, date and cloud cover
START, END = "2024-01-01", "2024-12-31"

landsat = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
             .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
             .filterBounds(AOI)                       # only scenes that touch the AOI
             .filterDate(START, END)                  # only one year
             .filter(ee.Filter.lt("CLOUD_COVER", 30)) # scenes with less than 30% cloud
             .sort("system:time_start"))              # sort by date

print("Number of scenes:", landsat.size().getInfo())

In [ ]:
# Look at the scenes as a table: date and cloud cover of each one
scenes = pd.DataFrame({
    "date": landsat.aggregate_array("DATE_ACQUIRED").getInfo(),
    "satellite": landsat.aggregate_array("SPACECRAFT_ID").getInfo(),
    "cloud_cover (%)": landsat.aggregate_array("CLOUD_COVER").getInfo(),
})
scenes.head(10)

### Scale factor and cloud mask

**Scale factor.** Landsat Level-2 stores reflectance as integers to save space. To get the
physical value (reflectance, roughly 0 to 1) we apply:

$$\rho = DN \times 0.0000275 - 0.2$$

Without this step EVI and MSAVI give wrong values, and thresholds like `NDVI > 0.3` lose
meaning when compared with other studies.

**Cloud mask.** The scene-level filter (`CLOUD_COVER < 30`) is not enough: a scene with 10%
cloud still has cloudy pixels. The `QA_PIXEL` band flags each pixel, bit by bit:

| Bit | Meaning |
|---|---|
| 1 | dilated cloud (cloud edge) |
| 2 | cirrus |
| 3 | cloud |
| 4 | cloud shadow |

We keep only the pixels where **none** of these bits is set, using `updateMask`.

In [ ]:
def mask_clouds(image):
    # QA_PIXEL: bits 1 (dilated cloud), 2 (cirrus), 3 (cloud) and 4 (shadow)
    qa = image.select("QA_PIXEL")
    cloud_bits = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)
    clear = qa.bitwiseAnd(cloud_bits).eq(0)      # 1 = clear pixel, 0 = cloud or shadow
    return image.updateMask(clear)               # cloudy pixels become "no data"


def scale_and_rename(image):
    # Apply the Collection 2 scale factor to the optical bands
    optical = image.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
    optical = optical.multiply(0.0000275).add(-0.2)
    # Give the bands readable names
    optical = optical.rename(["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"])
    # Keep the original properties (date, cloud cover...) of the image
    return ee.Image(optical.copyProperties(image, ["system:time_start", "CLOUD_COVER", "DATE_ACQUIRED"]))


# Apply both functions to every image of the collection
collection = landsat.map(mask_clouds).map(scale_and_rename)

print("Bands:", collection.first().bandNames().getInfo())

---

## 4.2 - Indices espectrales / Spectral indices as new bands

An index is just **another band**. We write one function that computes the five indices and
appends them to the image with `addBands`. Applied with `.map()`, every image of the
collection gets the new bands.

| Index | Formula | What it shows |
|---|---|---|
| **NDVI** | (NIR - Red) / (NIR + Red) | vegetation vigour |
| **MSAVI** | (2 NIR + 1 - sqrt((2 NIR + 1)^2 - 8 (NIR - Red))) / 2 | vegetation, reduces soil influence (sparse vegetation, like the altiplano) |
| **EVI** | 2.5 (NIR - Red) / (NIR + 6 Red - 7.5 Blue + 1) | vegetation, less saturation in dense canopies |
| **NDWI** | (Green - NIR) / (Green + NIR) | open water (McFeeters, 1996) |
| **NDMI** | (NIR - SWIR1) / (NIR + SWIR1) | moisture in vegetation and soil |

`normalizedDifference` covers the simple ratios. EVI and MSAVI need `expression`, which lets
us write the formula almost as on paper.

In [ ]:
def add_indices(image):
    # Normalised differences: (A - B) / (A + B)
    ndvi = image.normalizedDifference(["NIR", "Red"]).rename("NDVI")
    ndwi = image.normalizedDifference(["Green", "NIR"]).rename("NDWI")
    ndmi = image.normalizedDifference(["NIR", "SWIR1"]).rename("NDMI")

    # EVI: formula written with expression()
    evi = image.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": image.select("NIR"), "RED": image.select("Red"), "BLUE": image.select("Blue")},
    ).rename("EVI")

    # MSAVI: formula written with expression()
    msavi = image.expression(
        "(2 * NIR + 1 - sqrt((2 * NIR + 1) ** 2 - 8 * (NIR - RED))) / 2",
        {"NIR": image.select("NIR"), "RED": image.select("Red")},
    ).rename("MSAVI")

    # Add the five indices as new bands
    return image.addBands([ndvi, msavi, evi, ndwi, ndmi])


# Every image of the collection now has 6 optical bands + 5 indices
collection = collection.map(add_indices)

print("Bands:", collection.first().bandNames().getInfo())

This collection is our **data cube**: for each date (time) we have an image (x, y) with 11
bands. It is clean (no clouds), in physical units (reflectance) and enriched with indices.

---

## 4.3 - Una escena y estadisticas zonales / One scene and zonal statistics

We pick the **least cloudy scene** of the year. A Landsat scene may not cover the whole AOI,
so we take the date of that scene and **mosaic** all images acquired on that same day
(neighbouring scenes of the same orbit).

In [ ]:
# The least cloudy image and its date
best = collection.sort("CLOUD_COVER").first()
date = ee.Date(best.get("system:time_start"))

# Mosaic all images of that day and clip to the AOI
scene = collection.filterDate(date, date.advance(1, "day")).mosaic().clip(AOI)

print("Scene date:", date.format("YYYY-MM-dd").getInfo(),
      "| cloud cover:", round(best.get("CLOUD_COVER").getInfo(), 1), "%")

### Draw your region of interest (ROI)

Run the next cell and use the **drawing tools** on the left of the map to draw a rectangle or
polygon (for example over a crop field, the lake or the city). Then run the following cell.

If you do not draw anything, a default ROI (a 500 m circle on the Katari plain) is used.

In [ ]:
# Visualisation parameters
vis_rgb = {"bands": ["Red", "Green", "Blue"], "min": 0.0, "max": 0.3}    # natural colour
vis_false = {"bands": ["NIR", "Red", "Green"], "min": 0.0, "max": 0.4}   # false colour (vegetation in red)

# A new map for the scene
Map_scene = geemap.Map()
Map_scene.centerObject(AOI, 10)
Map_scene.addLayer(scene, vis_rgb, "RGB (natural colour)")
Map_scene.addLayer(scene, vis_false, "False colour (NIR-R-G)", False)
Map_scene

In [ ]:
# Take the geometry drawn on the map (None if nothing was drawn)
roi = Map_scene.user_roi

# Default ROI: a 500 m buffer around a point on the Katari plain
if roi is None:
    roi = ee.Geometry.Point([-68.55, -16.50]).buffer(500)

# Show the ROI on the map
Map_scene.addLayer(roi, {"color": "yellow"}, "ROI")
Map_scene.centerObject(roi, 14)

### Zonal statistics with `reduceRegion`

A ROI contains many pixels, and they **do not all have the same value**. `reduceRegion`
summarises them. Combining reducers we get, in one call, the **min, mean, max and standard
deviation** of every band inside the ROI.

We write it once as a function that returns a `DataFrame`, and a second function that draws
the chart. Then we use both for the optical bands, the vegetation indices and the water
indices.

In [ ]:
def zonal_stats(image, bands, region, scale=30):
    # One reducer that computes mean, min, max and standard deviation at once
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.minMax(), sharedInputs=True)
               .combine(ee.Reducer.stdDev(), sharedInputs=True))

    # Summarise the pixels of the region (30 m = Landsat pixel size)
    stats = image.select(bands).reduceRegion(
        reducer=reducer, geometry=region, scale=scale, maxPixels=1e9).getInfo()

    # Organise the result: one row per band, one column per statistic
    table = {s: [stats[f"{b}_{s}"] for b in bands] for s in ["min", "mean", "max", "stdDev"]}
    return pd.DataFrame(table, index=bands).round(3)


def plot_stats(df, title, ylabel, color):
    fig, ax = plt.subplots(figsize=(8, 4))
    # Bars: mean value, with the standard deviation as error bar
    ax.bar(df.index, df["mean"], yerr=df["stdDev"], capsize=5,
           color=color, alpha=0.8, label="mean ± std. dev.")
    # Markers: minimum and maximum pixel value in the ROI
    ax.scatter(df.index, df["min"], marker="v", color="black", zorder=3, label="min")
    ax.scatter(df.index, df["max"], marker="^", color="black", zorder=3, label="max")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Band")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()

#### Optical bands

Look at the shape of the bars from Blue to SWIR2: this is the **spectral signature** of the
ROI. Vegetation has low Red and high NIR; water is low everywhere and almost zero in NIR and
SWIR; bare soil rises gently from Blue to SWIR.

In [ ]:
optical_bands = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

df_optical = zonal_stats(scene, optical_bands, roi)
df_optical

In [ ]:
plot_stats(df_optical, "Optical bands in the ROI", "Surface reflectance", "#7f8c8d")

#### Vegetation indices

In [ ]:
veg_indices = ["NDVI", "MSAVI", "EVI"]

df_veg = zonal_stats(scene, veg_indices, roi)
df_veg

In [ ]:
plot_stats(df_veg, "Vegetation indices in the ROI", "Index value", "#2e8b57")

#### Water and moisture indices

In [ ]:
water_indices = ["NDWI", "NDMI"]

df_water = zonal_stats(scene, water_indices, roi)
df_water

In [ ]:
plot_stats(df_water, "Water and moisture indices in the ROI", "Index value", "#2c7bb6")

**How to read the charts.** The bar is the mean, the error bar the standard deviation, and
the triangles the min and max. A large distance between min and max means the ROI is **not
homogeneous** - it mixes surfaces (a field and its border, water and shore). This variability
is exactly what a classifier has to deal with.

Draw a new ROI over another surface, re-run the cells, and compare.

### The same with `geemap.zonal_stats`

geemap offers a ready-made function that does the same with `reduceRegions`, for a
`FeatureCollection` (one or many zones), and saves the result to CSV.

In [ ]:
# The ROI as a FeatureCollection (geemap.zonal_stats works with zones/polygons)
roi_fc = ee.FeatureCollection([ee.Feature(roi, {"zone": "ROI"})])

# Mean of every band and index in the ROI, saved to a CSV file
geemap.zonal_stats(scene, roi_fc, "zonal_stats_roi.csv", stat_type="MEAN", scale=30)

pd.read_csv("zonal_stats_roi.csv").round(3)

---

## 4.4 - El NDVI cambia en el tiempo / NDVI changes in time

A single scene is one moment. On the altiplano the vegetation grows in the rainy season
(December-March) and dries in winter (June-August). Reducing the whole **year** pixel by pixel
we get:

- **NDVI min** - the driest moment of each pixel
- **NDVI max** - the greenest moment of each pixel
- **Amplitude** = max - min - how much each pixel **changes** during the year

Water and urban areas change little (low amplitude). Crops change a lot (high amplitude).
This is why time is a powerful dimension of the data cube for classification.

In [ ]:
# NDVI band of every image of the year
ndvi_year = collection.select("NDVI")

# Pixel-by-pixel minimum, maximum and amplitude
ndvi_min = ndvi_year.min().clip(AOI)
ndvi_max = ndvi_year.max().clip(AOI)
ndvi_amp = ndvi_max.subtract(ndvi_min).rename("NDVI_amplitude")

# Colour palettes
vis_ndvi = {"min": -0.2, "max": 0.8, "palette": ["#a6611a", "#f5f5dc", "#a6d96a", "#1a9641", "#00441b"]}
vis_amp = {"min": 0.0, "max": 0.6, "palette": ["#ffffcc", "#fd8d3c", "#bd0026"]}

Map_ndvi = geemap.Map()
Map_ndvi.centerObject(AOI, 10)
Map_ndvi.addLayer(scene, vis_rgb, "RGB scene", False)
Map_ndvi.addLayer(ndvi_min, vis_ndvi, "NDVI min")
Map_ndvi.addLayer(ndvi_max, vis_ndvi, "NDVI max")
Map_ndvi.addLayer(ndvi_amp, vis_amp, "NDVI amplitude (max - min)")
Map_ndvi.addLayer(roi, {"color": "yellow"}, "ROI")
Map_ndvi.add_colorbar(vis_ndvi, label="NDVI (min / max)")
Map_ndvi.add_colorbar(vis_amp, label="NDVI amplitude")
Map_ndvi

Use the layer control (top right) to switch between **NDVI min** and **NDVI max**: the same
pixel has very different values depending on the date. The amplitude layer summarises that
difference in one image.

### NDVI over time in the ROI

To see it in a chart we compute the mean NDVI of the ROI for every image of the collection.

In [ ]:
def ndvi_in_roi(image):
    # Mean NDVI of the ROI for one image, stored as a property of a Feature
    mean = image.select("NDVI").reduceRegion(ee.Reducer.mean(), roi, 30).get("NDVI")
    return ee.Feature(None, {"date": image.date().format("YYYY-MM-dd"), "NDVI": mean})


# One Feature per image; drop the dates where the ROI was covered by clouds
series = ee.FeatureCollection(collection.map(ndvi_in_roi)).filter(ee.Filter.notNull(["NDVI"]))

df_series = pd.DataFrame({
    "date": pd.to_datetime(series.aggregate_array("date").getInfo()),
    "NDVI": series.aggregate_array("NDVI").getInfo(),
})

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df_series["date"], df_series["NDVI"], marker="o", color="#2e8b57", label="mean NDVI in the ROI")
ax.axhline(df_series["NDVI"].min(), color="#a6611a", linestyle="--", label="min")
ax.axhline(df_series["NDVI"].max(), color="#1a9641", linestyle="--", label="max")
ax.set_title(f"NDVI in the ROI during {START[:4]}")
ax.set_xlabel("Date")
ax.set_ylabel("NDVI")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

---

## 4.5 - Mascaras por umbral / Threshold masks

A mask turns a continuous index into a **yes / no** image:

- **Vegetation**: `NDVI > 0.3`
- **Water**: `NDWI > 0.1`

`gt()` returns 1 where the condition is true and 0 where it is false. Then:

- `selfMask()` hides the zeros, so only the class is drawn on top of the RGB image;
- `updateMask(mask)` applies the mask to **another** image - for example, to show the RGB
  colours only where there is vegetation.

These thresholds are a simple starting point. They vary with sensor, date and place - always
check them visually.

In [ ]:
# Binary masks (1 = class, 0 = not class)
veg_mask = scene.select("NDVI").gt(0.3).rename("vegetation")
water_mask = scene.select("NDWI").gt(0.1).rename("water")

Map_mask = geemap.Map()
Map_mask.centerObject(AOI, 10)

# RGB composition in the background
Map_mask.addLayer(scene, vis_rgb, "RGB (natural colour)")

# selfMask: draw only the pixels where the mask is 1
Map_mask.addLayer(veg_mask.selfMask(), {"palette": ["#1a9641"]}, "Vegetation (NDVI > 0.3)")
Map_mask.addLayer(water_mask.selfMask(), {"palette": ["#2c7bb6"]}, "Water (NDWI > 0.1)")

# updateMask: the RGB image only where there is vegetation
Map_mask.addLayer(scene.updateMask(veg_mask), vis_rgb, "RGB only on vegetation", False)

Map_mask.add_legend(title="Masks", legend_dict={"Vegetation": "#1a9641", "Water": "#2c7bb6"})
Map_mask

### How much area does each mask cover?

`ee.Image.pixelArea()` gives the area of each pixel in m2. Multiplying it by the mask (1 or
0) and summing over the AOI gives the area of each class.

In [ ]:
# Area of each pixel (m2) times the mask (1/0) -> area of the class in each pixel
areas = (veg_mask.addBands(water_mask)
                 .multiply(ee.Image.pixelArea())
                 .reduceRegion(reducer=ee.Reducer.sum(), geometry=AOI, scale=30, maxPixels=1e10)
                 .getInfo())

# m2 -> km2
df_area = pd.DataFrame({"area (km2)": areas}).divide(1e6).round(1)
df_area

---

## 4.6 - Resumen / Summary: our data cube

| Step | What we did | GEE tool |
|---|---|---|
| 1 | Study area and Landsat 8 + 9 collection | `filterBounds`, `filterDate`, `filter`, `merge` |
| 2 | Remove cloudy pixels | `QA_PIXEL`, `bitwiseAnd`, `updateMask` |
| 3 | Physical units | scale factor `* 0.0000275 - 0.2` |
| 4 | Indices as new bands | `normalizedDifference`, `expression`, `addBands` |
| 5 | Pixel variability in a ROI | `reduceRegion`, combined reducers, `geemap.zonal_stats` |
| 6 | Variability in time | `min()`, `max()` of the collection |
| 7 | Simple classes | `gt()`, `selfMask()`, `updateMask()` |

The final image (or the whole collection) with optical bands + indices is the input for the
next step: **supervised image classification**, where instead of choosing a threshold by hand,
an algorithm learns the rules from samples.

In [ ]:
# The bands of our prepared scene: the input for classification
scene.bandNames().getInfo()

---

## 4.7 - Ejercicio / Exercise

1. **Three ROIs.** Draw a ROI over water, one over cropland and one over the city. Run the
   zonal statistics for each and compare the bar charts. Which band separates the three best?
2. **Which vegetation index?** Compare NDVI, MSAVI and EVI in a sparse-vegetation area. Which
   one gives the lowest values, and why (think about the soil)?
3. **Change the thresholds.** Try `NDVI > 0.2` and `NDVI > 0.5`, and `NDWI > 0` and
   `NDWI > 0.3`. How does the area change? Where on the map do the differences appear?
4. **Another season.** Change the dates to the rainy season only (December-March). What
   happens to the number of images and to the NDVI amplitude?

**Entregable / Deliverable**: the notebook with your ROIs, charts and a short paragraph for
each question.

---

## Referencias / References

**Indices**

- Rouse, J.W. et al. (1974). Monitoring vegetation systems in the Great Plains with ERTS.
  *Third ERTS Symposium*, NASA SP-351, 309-317. (NDVI)
- Qi, J., Chehbouni, A., Huete, A.R., Kerr, Y.H., Sorooshian, S. (1994). A modified soil
  adjusted vegetation index. *Remote Sensing of Environment*, 48(2), 119-126. (MSAVI)
- Huete, A. et al. (2002). Overview of the radiometric and biophysical performance of the
  MODIS vegetation indices. *Remote Sensing of Environment*, 83(1-2), 195-213. (EVI)
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI) in the
  delineation of open water features. *International Journal of Remote Sensing*, 17(7),
  1425-1432. (NDWI)
- Gao, B.-C. (1996). NDWI - A normalized difference water index for remote sensing of
  vegetation liquid water from space. *Remote Sensing of Environment*, 58(3), 257-266.
  (the moisture index known today as NDMI)

**Earth Engine and Landsat**

- Landsat 8 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2>
- Landsat 9 C2 L2: <https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2>
- Landsat Collection 2 scale factors: <https://www.usgs.gov/faqs/how-do-i-use-scale-factor-landsat-level-2-science-products>
- Reducing regions: <https://developers.google.com/earth-engine/guides/reducers_reduce_region>
- Masking: <https://developers.google.com/earth-engine/guides/image_relational>
- geemap: <https://geemap.org>

**Book (open access)**

- Cardille, J.A., Crowley, M.A., Saah, D., Clinton, N.E. (eds.) (2023). *Cloud-Based Remote
  Sensing with Google Earth Engine: Fundamentals and Applications*. Springer.
  <https://www.eefabook.org/>